# Sales & Campaign Performance Analytics
**Objective:** Analyze campaign effectiveness across 10+ variants, flag underperformers, and surface KPI insights across 8+ metrics.

**Pipeline:** 4 raw sources → ETL → 50 campaigns × 30 KPI columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/campaign_master.csv', parse_dates=['start_date','end_date'])
print(f'Shape: {df.shape}')
df.head()

## 1. KPI Overview — 8 Core Metrics

In [ ]:
kpis = {
    'Total Campaigns':      len(df),
    'Total Spend ($)':      f"${df['total_spend'].sum():,.0f}",
    'Total Revenue ($)':    f"${df['total_revenue'].sum():,.0f}",
    'Overall ROAS':         f"{df['total_revenue'].sum()/df['total_spend'].sum():.2f}x",
    'Avg CTR':              f"{df['ctr'].mean():.2%}",
    'Avg CVR':              f"{df['cvr'].mean():.2%}",
    'Avg CPA ($)':          f"${df['cpa'].mean():.0f}",
    'Underperforming':      f"{df['underperforming_flags'].ge(2).sum()} campaigns",
}
print('=== KPI DASHBOARD ===')
for k, v in kpis.items():
    print(f'{k:<25} {v}')

## 2. Channel Performance Comparison

In [ ]:
ch = df.groupby('channel').agg(
    campaigns   =('campaign_id','count'),
    total_spend =('total_spend','sum'),
    total_rev   =('total_revenue','sum'),
    avg_ctr     =('ctr','mean'),
    avg_cvr     =('cvr','mean'),
    avg_roas    =('roas','mean'),
).round(3).reset_index()
ch['roas'] = (ch['total_rev'] / ch['total_spend']).round(2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#3498db','#2ecc71','#e74c3c','#9b59b6','#e67e22']

for ax, col, title, fmt in zip(axes,
    ['avg_ctr','avg_cvr','roas'],
    ['Avg CTR by Channel','Avg CVR by Channel','ROAS by Channel'],
    ['pct','pct','num']):
    vals = ch.set_index('channel')[col].sort_values(ascending=False)
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='white')
    for bar, val in zip(bars, vals.values):
        label = f'{val:.2%}' if fmt=='pct' else f'{val:.2f}x'
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../visuals/channel_performance.png', bbox_inches='tight')
plt.show()

## 3. Campaign Type Analysis

In [ ]:
ct = df.groupby('campaign_type').agg(
    avg_roas  =('roas','mean'),
    avg_cvr   =('cvr','mean'),
    avg_cpa   =('cpa','mean'),
    total_rev =('total_revenue','sum'),
).round(3).reset_index().sort_values('avg_roas', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].barh(ct['campaign_type'], ct['avg_roas'],
                    color='#3498db', edgecolor='white')
for bar, val in zip(bars, ct['avg_roas']):
    axes[0].text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                 f'{val:.2f}x', va='center', fontsize=10, fontweight='bold')
axes[0].axvline(x=1.5, color='red', linestyle='--', alpha=0.6, label='Min ROAS threshold (1.5x)')
axes[0].set_title('Avg ROAS by Campaign Type', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

bars2 = axes[1].barh(ct['campaign_type'], ct['avg_cpa'],
                     color='#e74c3c', edgecolor='white')
for bar, val in zip(bars2, ct['avg_cpa']):
    axes[1].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'${val:.0f}', va='center', fontsize=10, fontweight='bold')
axes[1].set_title('Avg CPA by Campaign Type', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../visuals/campaign_type_analysis.png', bbox_inches='tight')
plt.show()

## 4. Underperforming Campaign Flags

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Flag distribution
flag_counts = df['underperforming_flags'].value_counts().sort_index()
colors_flag = ['#2ecc71','#f1c40f','#e67e22','#e74c3c']
axes[0].bar(flag_counts.index, flag_counts.values,
            color=colors_flag[:len(flag_counts)], edgecolor='white')
axes[0].set_title('Distribution of Underperforming Flags per Campaign', fontweight='bold')
axes[0].set_xlabel('Number of Flags (0 = healthy, 3 = critical)')
axes[0].set_ylabel('# Campaigns')
for i, (idx, val) in enumerate(flag_counts.items()):
    axes[0].text(idx, val+0.2, str(val), ha='center', fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# Performance tier
tier_counts = df['performance_tier'].value_counts()
tier_colors = {'Poor':'#e74c3c','Below Average':'#e67e22','Good':'#3498db','Excellent':'#2ecc71'}
colors_tier = [tier_colors.get(t,'gray') for t in tier_counts.index]
axes[1].pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%',
            colors=colors_tier, startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Campaign Performance Tier Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('../visuals/underperformer_flags.png', bbox_inches='tight')
plt.show()

print('\nTop 10 Underperforming Campaigns:')
cols = ['campaign_id','channel','campaign_type','ctr','cvr','roas','underperforming_flags','performance_tier']
print(df[df['underperforming_flags']>=2][cols].sort_values('roas').head(10).to_string(index=False))

## 5. Spend vs Revenue Scatter — All 50 Campaigns

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
channel_colors = {'Email':'#3498db','Paid Search':'#e74c3c','Social Media':'#2ecc71',
                  'Display':'#9b59b6','Referral':'#e67e22'}

for channel, group in df.groupby('channel'):
    ax.scatter(group['total_spend'], group['total_revenue'],
               c=channel_colors[channel], label=channel,
               s=group['roas'].fillna(1)*40, alpha=0.75, edgecolors='white', linewidth=0.5)

# Break-even line (ROAS = 1)
max_val = max(df['total_spend'].max(), df['total_revenue'].max())
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Break-even (ROAS=1)')
ax.plot([0, max_val], [0, max_val*1.5], 'g--', alpha=0.3, label='ROAS=1.5 threshold')

ax.set_xlabel('Total Spend ($)', fontsize=11)
ax.set_ylabel('Total Revenue ($)', fontsize=11)
ax.set_title('Spend vs Revenue by Campaign (bubble size = ROAS)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax.legend(loc='upper left', fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../visuals/spend_vs_revenue.png', bbox_inches='tight')
plt.show()

## 6. KPI Heatmap — All Channels × Campaign Types

In [ ]:
pivot_roas = df.groupby(['channel','campaign_type'])['roas'].mean().unstack().round(2)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot_roas, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.3, ax=ax, center=1.5,
            annot_kws={'size':10})
ax.set_title('ROAS Heatmap: Channel × Campaign Type (red = below 1.5x threshold)', fontweight='bold', pad=12)
ax.set_xlabel('Campaign Type')
ax.set_ylabel('Channel')
plt.tight_layout()
plt.savefig('../visuals/roas_heatmap.png', bbox_inches='tight')
plt.show()

## 7. Monthly Spend & Revenue Trend

In [ ]:
df['month'] = df['start_date'].dt.to_period('M')
monthly = df.groupby('month').agg(
    spend   =('total_spend','sum'),
    revenue =('total_revenue','sum'),
    campaigns=('campaign_id','count')
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)
monthly['roas'] = (monthly['revenue']/monthly['spend']).round(2)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

x = range(len(monthly))
ax1.bar(x, monthly['spend'], width=0.4, align='center', color='#3498db', alpha=0.7, label='Spend')
ax1.bar([i+0.4 for i in x], monthly['revenue'], width=0.4, align='center', color='#2ecc71', alpha=0.7, label='Revenue')
ax2.plot([i+0.2 for i in x], monthly['roas'], 'ro-', linewidth=2, markersize=6, label='ROAS')

ax1.set_xticks([i+0.2 for i in x])
ax1.set_xticklabels(monthly['month_str'], rotation=45, ha='right')
ax1.set_ylabel('Amount ($)')
ax2.set_ylabel('ROAS')
ax1.set_title('Monthly Spend vs Revenue vs ROAS', fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax1.spines[['top']].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')

plt.tight_layout()
plt.savefig('../visuals/monthly_trend.png', bbox_inches='tight')
plt.show()

## 8. Key Findings & Recommendations

In [ ]:
best_channel  = df.groupby('channel')['roas'].mean().idxmax()
worst_channel = df.groupby('channel')['roas'].mean().idxmin()
best_type     = df.groupby('campaign_type')['roas'].mean().idxmax()
underperf_n   = df['underperforming_flags'].ge(2).sum()
total_waste   = df[df['roas']<1]['total_spend'].sum()

print('=' * 55)
print('EXECUTIVE SUMMARY')
print('=' * 55)
print(f'Best Channel      : {best_channel}')
print(f'Worst Channel     : {worst_channel}')
print(f'Best Campaign Type: {best_type}')
print(f'Underperformers   : {underperf_n} campaigns (2+ flags)')
print(f'Estimated Waste   : ${total_waste:,.0f} on ROAS < 1 campaigns')
print()
print('RECOMMENDATIONS')
print('-' * 55)
print(f'1. Reallocate budget from {worst_channel} to {best_channel}.')
print(f'   {best_channel} delivers highest ROAS consistently.')
print()
print(f'2. Pause {underperf_n} flagged campaigns immediately.')
print(f'   Estimated spend recovery: ${total_waste:,.0f}')
print()
print(f'3. Scale {best_type} campaigns — highest conversion')
print('   efficiency across all channels. Increase budget by 20%.')